In [14]:
import numpy as np, pandas as pd, torch
import torch.nn as nn
from sklearn.linear_model import LinearRegression

base = "../data"
FEATURES = ["rv_15m","rv_1h","rv_4h","rv_24h","parkinson_15m","parkinson_1h","volume_zscore","buy_ratio"]
TARGET, SEQ_LEN = "target_15m", 48

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
te = pd.read_parquet(f"{base}/scaled/test",
        columns=["symbol","event_time"]+FEATURES+[TARGET])
te = te.sort_values(["symbol","event_time"])
te = te.groupby("symbol", group_keys=False, observed=True).apply(lambda g: g.iloc[::5]).dropna()
print("rows:", len(te))

rows: 231918


/tmp/ipykernel_7424/1729678729.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  te = te.groupby("symbol", group_keys=False, observed=True).apply(lambda g: g.iloc[::5]).dropna()


In [16]:
def build_sequences_with_keys(df):
    Xs, ys, syms, times = [], [], [], []
    for sym, g in df.groupby("symbol", observed=True):
        if len(g) <= SEQ_LEN: continue
        feats = g[FEATURES].to_numpy(np.float32)
        tgt   = g[TARGET].to_numpy(np.float32)
        ts    = g["event_time"].to_numpy()
        w = np.lib.stride_tricks.sliding_window_view(feats, SEQ_LEN, axis=0).transpose(0,2,1)
        Xs.append(w[:-1]); ys.append(tgt[SEQ_LEN:])
        syms.append(np.full(len(w)-1, sym)); times.append(ts[SEQ_LEN:])
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(syms), np.concatenate(times)

Xte, yte, symte, timete = build_sequences_with_keys(te)
print("Xte:", Xte.shape)

Xte: (230958, 48, 8)


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(f"{base}/lstm_model.pt", map_location=device, weights_only=False)
cfg, sym2id = ckpt["config"], ckpt["sym2id"]
y_mean, y_std = ckpt["y_mean"], ckpt["y_std"]

class VolLSTM(nn.Module):
    def __init__(s, n_features, n_symbols, emb_dim, hidden):
        super().__init__()
        s.emb = nn.Embedding(n_symbols, emb_dim)
        s.lstm = nn.LSTM(n_features+emb_dim, hidden, num_layers=2, dropout=0.2, batch_first=True)
        s.head = nn.Linear(hidden, 1)
    def forward(s, x, sid):
        e = s.emb(sid).unsqueeze(1).expand(-1, x.size(1), -1)
        out, _ = s.lstm(torch.cat([x, e], dim=2))
        return s.head(out[:, -1, :]).squeeze(1)

model = VolLSTM(cfg["n_features"], cfg["n_symbols"], cfg["emb_dim"], cfg["hidden"]).to(device)
model.load_state_dict(ckpt["model_state"]); model.eval()

idte = np.array([sym2id[s] for s in symte], dtype=np.int64)
preds = []
with torch.no_grad():
    for i in range(0, len(Xte), 4096):
        xb = torch.from_numpy(Xte[i:i+4096]).to(device)
        ib = torch.from_numpy(idte[i:i+4096]).to(device)
        preds.append(model(xb, ib).cpu().numpy())
lstm_pred = np.concatenate(preds) * y_std + y_mean
print("predicted:", lstm_pred.shape)

predicted: (230958,)


In [18]:
res = pd.DataFrame({"symbol": symte, "event_time": timete,
                    "y_true": yte, "lstm_pred": lstm_pred})

raw = pd.read_parquet(f"{base}/klines_processed/features_with_targets",
        columns=["symbol","event_time","rv_15m","rv_1h","rv_24h"])
raw = raw.drop_duplicates(subset=["symbol","event_time"])
for c in ["rv_15m","rv_1h","rv_24h"]:
    raw[c] = np.log1p(raw[c])

res = res.merge(raw, on=["symbol","event_time"], how="left")
print("rows:", len(res), "(must be 230958)")

rows: 230958 (must be 230958)


In [20]:
tr = pd.read_parquet(f"{base}/klines_processed/features_with_targets",
        columns=["symbol","event_time","rv_15m","rv_1h","rv_24h","target_15m"])
tr = tr.drop_duplicates(subset=["symbol","event_time"])
t0, t1 = tr.event_time.min(), tr.event_time.max(); span = t1 - t0
tr = tr[tr.event_time < t0 + span*0.8].copy()
for c in ["rv_15m","rv_1h","rv_24h"]:
    tr[c] = np.log1p(tr[c])

tr = tr.dropna(subset=["rv_15m","rv_1h","rv_24h","target_15m"])

har = LinearRegression().fit(tr[["rv_15m","rv_1h","rv_24h"]], tr["target_15m"])
res["har_pred"] = har.predict(res[["rv_15m","rv_1h","rv_24h"]])
print("har coef:", dict(zip(["rv_15m","rv_1h","rv_24h"], har.coef_)))

har coef: {'rv_15m': np.float64(0.16732474788682988), 'rv_1h': np.float64(0.009128281423598323), 'rv_24h': np.float64(0.00032560033442801447)}


In [21]:
def rmse(y,p): return np.sqrt(np.mean((y-p)**2))
def mae(y,p):  return np.mean(np.abs(y-p))

y = res["y_true"].to_numpy()
mean_pred = tr["target_15m"].mean()

print(f"{'model':<8} {'RMSE':>12} {'MAE':>12}")
for name, col in [("LSTM","lstm_pred"), ("HAR","har_pred")]:
    p = res[col].to_numpy()
    print(f"{name:<8} {rmse(y,p):>12.3e} {mae(y,p):>12.3e}")
print(f"{'mean':<8} {rmse(y,mean_pred):>12.3e} {mae(y,mean_pred):>12.3e}")

model            RMSE          MAE
LSTM        5.700e-05    1.705e-05
HAR         3.915e-05    1.860e-05
mean        4.433e-05    2.259e-05
